# Sesión 5 — MUBD

## Clustering no supervisado

Exploramos agrupaciones jerárquicas, k-means, k-medoides, CLARA y k-means con kernel.
La información de especie de Iris se reserva para examinar las agrupaciones después de construirlas.

En VS Code, selecciona un kernel de Python y ejecuta en orden con **Shift + Enter**.
Abre **Variables** para inspeccionar las tablas y **Esquema / Outline** para navegar.
Los controles se activan al ejecutar sus celdas. Usa **Reiniciar** y **Ejecutar todo** para repetir el análisis.
[Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

### Preparación

Vaciamos las variables de la sesión y cargamos las librerías.
Si falta alguna en el kernel seleccionado, descomenta la línea de instalación y ejecútala una vez.
Las funciones de apoyo están incluidas en el propio notebook; puedes desplegar sus celdas para leerlas.

In [ ]:
%reset -f

In [ ]:
# %pip install numpy pandas matplotlib scipy scikit-learn seaborn ipywidgets threadpoolctl ipykernel

In [ ]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Rectangle, Circle, Wedge
from pandas.plotting import scatter_matrix
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet, cut_tree
from scipy.spatial.distance import pdist, squareform, cdist
from scipy.spatial import ConvexHull
from scipy.linalg import eigh
from scipy.optimize import linear_sum_assignment
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from threadpoolctl import threadpool_limits

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})
pd.set_option("display.max_columns", 30)
SEED = 12345

Las funciones de k-means permiten elegir **Lloyd** o **Hartigan–Wong**, con uno o varios inicios.
En Hartigan–Wong se alternan transferencias óptimas y rápidas de observaciones entre grupos.
Las etiquetas de grupo se numeran por orden de aparición y no tienen un significado de clase.

In [ ]:
def renumerar(etiquetas):
    equivalencias = {valor: i for i, valor in enumerate(dict.fromkeys(np.asarray(etiquetas).tolist()))}
    return np.array([equivalencias[valor] for valor in etiquetas], dtype=int)


def escalar(tabla):
    return (tabla - tabla.mean()) / tabla.std(ddof=1)


def resumen_particion(datos, etiquetas):
    a = np.asarray(datos, dtype=float)
    grupos = renumerar(etiquetas)
    centros = np.array([a[grupos == i].mean(axis=0) for i in np.unique(grupos)])
    errores = np.sum((a - centros[grupos]) ** 2, axis=1)
    withinss = np.bincount(grupos, weights=errores)
    totss = np.sum((a - a.mean(axis=0)) ** 2)
    return SimpleNamespace(labels_=grupos, cluster_centers_=centros, withinss=withinss,
                           inertia_=withinss.sum(), totss=totss,
                           betweenss=totss - withinss.sum(), size=np.bincount(grupos))


def hartigan_wong(datos, k, n_init=1, random_state=12345, max_iter=100, init=None):
    """Transferencias óptimas y rápidas para minimizar la suma de cuadrados."""
    a = np.asarray(datos, dtype=float)
    rng = np.random.default_rng(random_state)
    unicos = np.unique(a, axis=0)
    if not 1 <= k <= len(unicos):
        raise ValueError("k debe estar entre 1 y el número de observaciones distintas.")
    mejor = None
    for inicio in range(n_init if init is None else 1):
        centros = unicos[rng.choice(len(unicos), k, replace=False)].copy() if init is None else np.asarray(init, dtype=float).copy()
        etiquetas = cdist(a, centros, metric="sqeuclidean").argmin(axis=1)
        tamanos = np.bincount(etiquetas, minlength=k).astype(float)
        if (tamanos == 0).any():
            raise ValueError("La inicialización produce un grupo vacío.")
        centros = np.array([a[etiquetas == j].mean(axis=0) for j in range(k)])
        alternativos = etiquetas.copy()

        def transferir(i, destino):
            origen = etiquetas[i]
            centros[origen] = (tamanos[origen] * centros[origen] - a[i]) / (tamanos[origen] - 1)
            centros[destino] = (tamanos[destino] * centros[destino] + a[i]) / (tamanos[destino] + 1)
            tamanos[origen] -= 1
            tamanos[destino] += 1
            etiquetas[i] = destino
            alternativos[i] = origen

        for iteracion in range(max_iter):
            cambios = 0
            # Etapa óptima: examinar todos los destinos posibles.
            for i, punto in enumerate(a):
                origen = etiquetas[i]
                if tamanos[origen] <= 1 or k == 1:
                    continue
                distancias = np.sum((centros - punto) ** 2, axis=1)
                retirada = tamanos[origen] / (tamanos[origen] - 1) * distancias[origen]
                costes = tamanos / (tamanos + 1) * distancias
                costes[origen] = np.inf
                destino = int(costes.argmin())
                alternativos[i] = destino
                if costes[destino] < retirada - 1e-12:
                    transferir(i, destino)
                    cambios += 1
            if cambios == 0:
                break
            # Etapa rápida: probar el último par de grupos candidato.
            for pasada in range(max_iter):
                rapidos = 0
                for i, punto in enumerate(a):
                    origen, destino = etiquetas[i], alternativos[i]
                    if origen == destino or tamanos[origen] <= 1:
                        continue
                    retirada = tamanos[origen] / (tamanos[origen] - 1) * np.sum((punto - centros[origen]) ** 2)
                    entrada = tamanos[destino] / (tamanos[destino] + 1) * np.sum((punto - centros[destino]) ** 2)
                    if entrada < retirada - 1e-12:
                        transferir(i, destino)
                        rapidos += 1
                if rapidos == 0:
                    break
        resultado = resumen_particion(a, etiquetas)
        resultado.n_iter_ = iteracion + 1
        if mejor is None or resultado.inertia_ < mejor.inertia_:
            mejor = resultado
    return mejor


def ajustar_kmeans(datos, k, n_init=10, random_state=12345, algoritmo="hartigan-wong", init=None):
    with threadpool_limits(limits=1):
        if algoritmo == "hartigan-wong":
            return hartigan_wong(datos, k, n_init, random_state, init=init)
        if algoritmo != "lloyd":
            raise ValueError("Algoritmo desconocido.")
        inicial = "random" if init is None else np.asarray(init)
        return KMeans(n_clusters=k, n_init=n_init if init is None else 1, init=inicial,
                      algorithm="lloyd", random_state=random_state, max_iter=300).fit(datos)

### Herramientas para representar grupos

La representación por componentes principales proyecta los datos sobre dos ejes.
Las envolventes convexas delimitan las observaciones de cada grupo.

In [ ]:
def representar_clusters(datos, etiquetas, titulo="Agrupación", envolventes=True, estandarizar=True):
    tabla = pd.DataFrame(datos)
    entrada = escalar(tabla) if estandarizar else tabla
    pca = PCA(n_components=2)
    coordenadas = pca.fit_transform(entrada)
    fig, ax = plt.subplots(figsize=(8, 5))
    colores = plt.get_cmap("tab10")
    for grupo in np.unique(etiquetas):
        puntos = coordenadas[np.asarray(etiquetas) == grupo]
        color = colores(int(grupo) % 10)
        ax.scatter(puntos[:, 0], puntos[:, 1], s=22, color=color, label=f"Grupo {int(grupo)+1}")
        if envolventes and len(puntos) >= 3 and np.linalg.matrix_rank(puntos - puntos.mean(axis=0)) == 2:
            borde = ConvexHull(puntos).vertices
            ax.fill(puntos[borde, 0], puntos[borde, 1], color=color, alpha=0.15)
    porcentajes = 100 * pca.explained_variance_ratio_
    ax.set(xlabel=f"CP1 ({porcentajes[0]:.1f} %)", ylabel=f"CP2 ({porcentajes[1]:.1f} %)", title=titulo)
    ax.legend()
    plt.show()

def pares_coloreados(tabla, etiquetas):
    colores = [plt.get_cmap("tab10")(int(grupo) % 10) for grupo in etiquetas]
    scatter_matrix(tabla, c=colores, figsize=(9, 9), diagonal="hist", s=15, alpha=0.7)
    plt.show()

## Inspeccionar los datos

In [ ]:
dataset = load_iris()
iris = pd.DataFrame(dataset.data, columns=["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"])
iris.index = np.arange(1, len(iris) + 1)
iris["Species"] = pd.Categorical.from_codes(dataset.target, dataset.target_names)
iris

In [ ]:
iris2 = iris.iloc[:, :4].copy()
iris2

#### Gráficos de pares

In [ ]:
_ = scatter_matrix(iris2, figsize=(9, 9), diagonal="hist", s=15, alpha=0.6)
plt.show()

#### Mapas de calor

Explora los datos originales y los estandarizados por columna, con media cero y desviación típica uno.
Los árboles del mapa reordenan filas y columnas según su proximidad.

In [ ]:
iris2_escalado = escalar(iris2)

def mapa_iris(estandarizado):
    tabla = iris2_escalado if estandarizado else iris2
    grafico = sns.clustermap(tabla, method="complete", cmap="vlag", figsize=(9, 7),
                            yticklabels=False, cbar_kws={"label": "Valor estandarizado" if estandarizado else "Valor"})
    grafico.fig.suptitle("Iris estandarizado" if estandarizado else "Iris sin escalar", y=1.02)
    plt.show()

In [ ]:
control_mapa = widgets.interactive(
    mapa_iris,
    estandarizado=widgets.Checkbox(value=False, description="Estandarizar columnas"),
)
display(control_mapa)

## Clustering jerárquico

### 1. Matriz de distancias

In [ ]:
d = pdist(iris2, metric="euclidean")
matriz_distancias = pd.DataFrame(squareform(d), index=iris2.index, columns=iris2.index)
matriz_distancias

### 2. Jerarquización: enlace completo

In [ ]:
hc = linkage(d, method="complete")

### 3. Dendrograma

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
_ = dendrogram(hc, labels=iris2.index.to_list(), leaf_rotation=90, leaf_font_size=5, ax=ax)
_ = ax.set(xlabel="Observación", ylabel="Distancia", title="Clustering jerárquico: enlace completo")
fig.tight_layout()
plt.show()

### 4. Calidad del árbol

La correlación cofenética resume cómo conserva el árbol las disimilitudes originales.

In [ ]:
cop_dist = cophenet(hc)
np.corrcoef(d, cop_dist)[0, 1]

### 5. Elegir una partición

#### Regla del codo: inercia dentro de los grupos

In [ ]:
inercia_hc = pd.Series({k: resumen_particion(iris2, cut_tree(hc, n_clusters=k).ravel()).inertia_
                        for k in range(1, 11)}, name="Inercia intra")
ax = inercia_hc.plot(marker="o")
_ = ax.set(xlabel="Número de grupos", ylabel="Inercia intra", title="Regla del codo")
plt.show()

#### Número de grupos según 26 indicadores

Calculamos KL, CH, Hartigan, CCC, Scott, Marriott, TraceCovW, TraceW, Friedman, Rubin,
C-index, Davies–Bouldin, silhouette, Duda, pseudo-t², Beale, Ratkowsky, Ball, punto-biserial,
Frey, McClain, Dunn, Hubert, SD, D-index y SDbw.

Cada criterio aplica su regla de elección. **Hubert y D-index son diagnósticos gráficos** y no emiten voto.
Las pruebas de división estudian una división local en dos del grupo de mayor inercia.
Los criterios sin recomendación válida se muestran como ausentes y no votan.

In [ ]:
def indice_dunn(distancias, etiquetas):
    D = np.asarray(distancias)
    grupos = np.unique(etiquetas)
    diametro = max(D[np.ix_(etiquetas == g, etiquetas == g)].max() for g in grupos)
    separacion = min(D[np.ix_(etiquetas == g, etiquetas == h)].min()
                     for i, g in enumerate(grupos) for h in grupos[i+1:])
    return separacion / diametro if diametro > 0 else np.inf


def evaluar_indices(datos, metodo="ward", min_k=2, max_k=15, random_state=12345):
    """26 criterios: 24 reglas de elección y dos diagnósticos gráficos."""
    a = np.asarray(datos, dtype=float)
    n, p = a.shape
    D = squareform(pdist(a))
    superiores = np.triu_indices(n, 1)
    distancias = D[superiores]
    ordenadas = np.sort(distancias)
    particiones = {1: np.zeros(n, dtype=int)}
    arbol = None if metodo == "kmeans" else linkage(pdist(a), method=metodo)
    for k in range(2, max_k + 2):
        particiones[k] = (ajustar_kmeans(a, k, n_init=1, random_state=random_state).labels_
                          if metodo == "kmeans" else renumerar(cut_tree(arbol, n_clusters=k).ravel()))
    total_centrado = np.sum((a - a.mean(axis=0)) ** 2)
    T = a.T @ a
    s = np.sqrt(np.maximum(np.linalg.eigvalsh(T / (n - 1))[::-1], 1e-15))
    determinante_T = np.linalg.slogdet(T)[1]
    cache = {}
    for k, etiquetas in particiones.items():
        r = resumen_particion(a, etiquetas)
        etiquetas = r.labels_
        residuos = a - r.cluster_centers_[etiquetas]
        W = residuos.T @ residuos
        B = T - W
        mismas = etiquetas[superiores[0]] == etiquetas[superiores[1]]
        dentro, fuera = distancias[mismas], distancias[~mismas]
        var_grupos = np.array([a[etiquetas == j].var(axis=0) for j in range(k)])
        norma_var = np.linalg.norm(var_grupos, axis=1)
        scatt = norma_var.mean() / np.linalg.norm(a.var(axis=0))
        radio = np.sqrt(norma_var.sum()) / k
        centros_D = squareform(pdist(r.cluster_centers_))
        dis = ((pdist(r.cluster_centers_).max() / pdist(r.cluster_centers_).min())
               * np.sum(1 / centros_D.sum(axis=1))) if k > 1 else np.nan
        cache[k] = dict(r=r, labels=etiquetas, W=W, B=B, dentro=dentro, fuera=fuera,
                        scatt=scatt, radio=radio, dis=dis, centros_D=centros_D)

    filas, criticos = [], []
    for k in range(min_k, max_k + 1):
        c = cache[k]
        r, etiquetas, W, B = c["r"], c["labels"], c["W"], c["B"]
        previo, siguiente = cache[k-1]["r"], cache[k+1]["r"]
        dentro, fuera = c["dentro"], c["fuera"]
        intra = r.inertia_
        signo, logdetW = np.linalg.slogdet(W)
        dispersion = np.sqrt(r.withinss / r.size)
        ratios = (dispersion[:, None] + dispersion[None, :]) / np.where(c["centros_D"] > 0, c["centros_D"], np.inf)
        m = len(dentro)
        minimo, maximo = ordenadas[:m].sum(), ordenadas[-m:].sum()
        u = s / (np.prod(s) / k) ** (1 / p)
        p1 = min(int((u >= 1).sum()), k - 1)
        if 0 < p1 < p:
            u = s / (np.prod(s[:p1]) / k) ** (1 / p1)
            suma = np.sum(1 / (n + u[:p1])) + np.sum(u[p1:] ** 2 / (n + u[p1:]))
            dimension = p1
        else:
            suma, dimension = np.sum(1 / (n + u)), p
        esperado_R2 = 1 - suma / np.sum(u ** 2) * (n - k) ** 2 / n * (1 + 4 / n)
        R2 = 1 - np.trace(W) / np.trace(T)
        ccc = (np.log((1 - esperado_R2) / (1 - R2)) * np.sqrt(n * dimension / 2)
               / (0.001 + esperado_R2) ** 1.2)

        # Pruebas de división: examinar una división local del grupo más disperso.
        padre = int(r.withinss.argmax())
        local = a[etiquetas == padre]
        division = ajustar_kmeans(local, 2, n_init=1, random_state=random_state)
        wl = division.inertia_
        wm = r.withinss[padre]
        nm = len(local)
        duda = wl / wm
        critica_duda = 1 - 2 / (np.pi * p) - 3.2 * np.sqrt(2 * (1 - 8 / np.pi ** 2) / (nm * p))
        pseudo = (wm - wl) / (wl / (nm - 2))
        beale = ((wm - wl) / wl) / ((nm - 1) / (nm - 2) * (2 ** (2 / p) - 1))
        criticos.append({"k": k, "Duda": critica_duda,
                         "PseudoT2": (1 - critica_duda) / critica_duda * (nm - 2),
                         "Beale": stats.f.sf(beale, p, (nm - 2) * p)})

        dist_centro = np.linalg.norm(a - r.cluster_centers_[etiquetas], axis=1)
        densidades = np.bincount(etiquetas, weights=(dist_centro <= c["radio"]), minlength=k)
        dens_bw = 0.0
        for i in range(k):
            for j in range(i+1, k):
                candidatos = a[(etiquetas == i) | (etiquetas == j)]
                centro = (r.cluster_centers_[i] + r.cluster_centers_[j]) / 2
                cantidad = (np.linalg.norm(candidatos - centro, axis=1) <= c["radio"]).sum()
                denominador = max(densidades[i], densidades[j])
                dens_bw += cantidad / denominador if denominador else 0
        Q = cdist(r.cluster_centers_[etiquetas], r.cluster_centers_[etiquetas])
        varP = np.linalg.norm(D.var(axis=0))
        varQ = np.linalg.norm(Q.var(axis=0))
        hubert = np.sum((D - D.mean())[superiores] * (Q - Q.mean())[superiores]) / (len(distancias) * varP * varQ)
        proxima = cache[k+1]
        denominador_frey = proxima["dentro"].mean() - dentro.mean()
        kl_num = (k-1) ** (2/p) * previo.inertia_ - k ** (2/p) * intra
        kl_den = k ** (2/p) * intra - (k+1) ** (2/p) * siguiente.inertia_
        filas.append({
            "k": k, "KL": abs(kl_num / kl_den),
            "CH": ((total_centrado - intra) / (k-1)) / (intra / (n-k)),
            "Hartigan": (intra / siguiente.inertia_ - 1) * (n-k), "CCC": ccc,
            "Scott": n * (determinante_T - logdetW), "Marriott": k ** 2 * np.linalg.det(W),
            "TraceCovW": np.var(W, axis=0, ddof=1).sum(), "TraceW": np.trace(W),
            "Friedman": np.trace(np.linalg.inv(W) * B), "Rubin": np.trace(T) / np.trace(W),
            "Cindex": (dentro.sum() - minimo) / (maximo - minimo), "DB": ratios.max(axis=1).mean(),
            "Silhouette": silhouette_score(D, etiquetas, metric="precomputed"),
            "Duda": duda, "PseudoT2": pseudo, "Beale": beale,
            "Ratkowsky": np.sqrt(1 - np.sum((a - r.cluster_centers_[etiquetas]) ** 2, axis=0)
                                    / np.sum((a - a.mean(axis=0)) ** 2, axis=0)).mean() / np.sqrt(k),
            "Ball": intra / k,
            "PointBiserial": (fuera.mean() - dentro.mean()) * np.sqrt(len(fuera)*len(dentro))
                              / len(distancias) / distancias.std(ddof=1),
            "Frey": (proxima["fuera"].mean() - fuera.mean()) / denominador_frey,
            "McClain": (dentro.mean()/len(dentro)) / (fuera.mean()/len(fuera)),
            "Dunn": indice_dunn(D, etiquetas), "Hubert": hubert,
            "SDindex": cache[max_k]["dis"] * c["scatt"] + c["dis"],
            "Dindex": dist_centro.mean(), "SDbw": c["scatt"] + 2 * dens_bw / (k * (k-1)),
        })
    indices = pd.DataFrame(filas).set_index("k")
    criticos = pd.DataFrame(criticos).set_index("k")
    recomendaciones = {}
    maximos = ["KL", "CH", "CCC", "Silhouette", "Ratkowsky", "PointBiserial", "Dunn"]
    minimos = ["Cindex", "DB", "McClain", "SDindex", "SDbw"]
    for nombre in maximos:
        recomendaciones[nombre] = indices[nombre].idxmax()
    for nombre in minimos:
        recomendaciones[nombre] = indices[nombre].idxmin()
    for nombre in ["Hartigan", "Scott", "TraceCovW", "Friedman", "Ball"]:
        recomendaciones[nombre] = (indices[nombre] - indices[nombre].shift(-1)).abs().idxmax()
    for nombre, maximizar in [("Marriott", True), ("TraceW", True), ("Rubin", False)]:
        diferencias = indices[nombre].shift(1) - 2 * indices[nombre] + indices[nombre].shift(-1)
        recomendaciones[nombre] = diferencias.idxmax() if maximizar else diferencias.idxmin()
    condiciones = {"Duda": indices["Duda"] >= criticos["Duda"],
                   "PseudoT2": indices["PseudoT2"] <= criticos["PseudoT2"],
                   "Beale": criticos["Beale"] >= 0.1, "Frey": indices["Frey"] < 1}
    for nombre, condicion in condiciones.items():
        candidatos = indices.index[condicion]
        recomendaciones[nombre] = (max(min_k, int(candidatos[0]) - 1) if nombre == "Frey" else candidatos[0]) if len(candidatos) else np.nan
    best = pd.Series(recomendaciones, name="k recomendado").astype("Int64")
    votos = best.dropna().value_counts().sort_index()
    ganador = int(votos.idxmax())
    return SimpleNamespace(All_index=indices, Best_n=best.to_frame(), Votes=votos,
                           Best_partition=particiones[ganador], Critical=criticos, partitions=particiones)

In [ ]:
NB = evaluar_indices(iris2, metodo="ward", min_k=2, max_k=15, random_state=SEED)
NB.Best_n

In [ ]:
ax = NB.Votes.plot.bar(rot=0)
_ = ax.set(xlabel="Número de grupos", ylabel="Votos", title="Recomendaciones de los indicadores")
plt.show()

#### Cortar el árbol en tres grupos

El control permite explorar otras particiones. Los rectángulos señalan las ramas que forman cada grupo.

In [ ]:
ct = pd.Series(renumerar(cut_tree(hc, n_clusters=3).ravel()), index=iris.index, name="grupo")

def particion_jerarquica(k):
    grupos = renumerar(cut_tree(hc, n_clusters=k).ravel())
    fig, ax = plt.subplots(figsize=(14, 5))
    dibujado = dendrogram(hc, no_labels=True, link_color_func=lambda nodo: "#444444", ax=ax)
    orden = np.asarray(dibujado["leaves"])
    altura = (hc[-k, 2] + hc[-k+1, 2]) / 2
    for grupo in np.unique(grupos):
        posiciones = np.flatnonzero(grupos[orden] == grupo)
        color = plt.get_cmap("tab10")(int(grupo) % 10)
        ax.add_patch(Rectangle((10 * posiciones.min(), 0), 10 * len(posiciones), altura,
                               fill=False, edgecolor=color, linewidth=2))
    ax.set(xlabel="Observaciones ordenadas por el árbol", ylabel="Distancia", title=f"Partición en {k} grupos")
    plt.show()
    resumen = resumen_particion(iris2, grupos)
    print(f"Variabilidad explicada: {resumen.betweenss / resumen.totss:.1%}")
    display(pd.Series(np.bincount(grupos), index=np.arange(1, k+1), name="Tamaño del grupo"))

In [ ]:
control_jerarquia = widgets.interactive(
    particion_jerarquica,
    k=widgets.IntSlider(value=3, min=2, max=8, description="Grupos", continuous_update=False),
)
display(control_jerarquia)

### Clustering sobre componentes principales y consolidación

El procedimiento reúne estandarización, componentes principales, jerarquización y una consolidación
con k-means iniciada en los centros de los grupos jerárquicos. Elige el número de grupos y activa la consolidación.

In [ ]:
def hcpc(k, consolidar):
    coordenadas = PCA().fit_transform(iris2_escalado)
    arbol = linkage(pdist(coordenadas), method="complete")
    inicial = renumerar(cut_tree(arbol, n_clusters=k).ravel())
    centros = np.array([coordenadas[inicial == j].mean(axis=0) for j in range(k)])
    grupos = ajustar_kmeans(coordenadas, k, init=centros).labels_ if consolidar else inicial
    representar_clusters(iris2_escalado, grupos, titulo="Clustering sobre componentes principales", estandarizar=False)
    return SimpleNamespace(labels_=grupos, tree_=arbol, scores_=coordenadas)

def explorar_hcpc(k, consolidar):
    global HC
    HC = hcpc(k, consolidar)

In [ ]:
control_hcpc = widgets.interactive(
    explorar_hcpc,
    k=widgets.IntSlider(value=3, min=2, max=8, description="Grupos", continuous_update=False),
    consolidar=widgets.Checkbox(value=True, description="Consolidar con k-means"),
)
display(control_hcpc)

### 6. Representación de los tres grupos jerárquicos

In [ ]:
pares_coloreados(iris2, ct.to_numpy())

#### Dos primeras componentes principales de las variables originales

In [ ]:
pr = PCA().fit(iris2)
scores = pr.transform(iris2)
x, y = scores[:, 0], scores[:, 1]
fig, ax = plt.subplots()
_ = ax.scatter(x, y, c=ct, cmap="tab10", s=24)
_ = ax.set(xlabel="CP1", ylabel="CP2", title="Clustering jerárquico en componentes principales")
plt.show()

#### Envolventes convexas sobre las dos primeras componentes de las variables estandarizadas

In [ ]:
representar_clusters(iris2, ct.to_numpy(), titulo="Tres grupos jerárquicos")

### 7. Correspondencia con las especies conocidas

Las especies se usan únicamente para evaluar la partición. Asignamos los grupos a especies
mediante la correspondencia de mayor coincidencia, porque los números de grupo son arbitrarios.

In [ ]:
tabla_especies = pd.crosstab(iris["Species"], ct + 1)
tabla_especies

In [ ]:
filas, columnas = linear_sum_assignment(-tabla_especies.to_numpy())
correspondencia = {int(columna): int(fila) for fila, columna in zip(filas, columnas)}
cl = ct.map(correspondencia)
t_pred = pd.crosstab(iris["Species"], cl + 1)
t_pred
np.trace(t_pred.to_numpy()) / t_pred.to_numpy().sum()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
marcadores = ["o", "s", "^"]
for i, nombre_y in enumerate(iris2.columns):
    for j, nombre_x in enumerate(iris2.columns):
        ax = axes[i, j]
        if i == j:
            _ = ax.hist(iris2[nombre_x], bins=12, color="lightgrey")
        else:
            for especie, marcador in enumerate(marcadores):
                mascara = dataset.target == especie
                _ = ax.scatter(iris2.loc[mascara, nombre_x], iris2.loc[mascara, nombre_y],
                               c=cl.loc[mascara], cmap="tab10", vmin=0, vmax=9, marker=marcador, s=12)
        if i == 3: _ = ax.set_xlabel(nombre_x)
        if j == 0: _ = ax.set_ylabel(nombre_y)
fig.tight_layout()
plt.show()

### 8. Relación entre dos dendrogramas

Construimos árboles con enlace completo y promedio. Las líneas unen la misma observación en ambos árboles.

In [ ]:
hc1 = linkage(d, method="complete")
hc2 = linkage(d, method="average")

def tanglegram(izquierdo, derecho):
    fig, axes = plt.subplots(1, 3, figsize=(14, 7), gridspec_kw={"width_ratios": [2, 1, 2]})
    a = dendrogram(izquierdo, orientation="left", no_labels=True, ax=axes[0], color_threshold=0)
    b = dendrogram(derecho, orientation="right", no_labels=True, ax=axes[2], color_threshold=0)
    posiciones_a = {observacion: 5 + 10 * i for i, observacion in enumerate(a["leaves"])}
    posiciones_b = {observacion: 5 + 10 * i for i, observacion in enumerate(b["leaves"])}
    for observacion in posiciones_a:
        axes[1].plot([0, 1], [posiciones_a[observacion], posiciones_b[observacion]], color="steelblue", alpha=0.3, linewidth=0.6)
    for ax in axes:
        ax.set_ylim(0, 10 * len(a["leaves"]))
    axes[1].set(xlim=(0, 1))
    axes[1].axis("off")
    axes[0].set_title("Enlace completo")
    axes[2].set_title("Enlace promedio")
    fig.tight_layout()
    plt.show()

tanglegram(hc1, hc2)

In [ ]:
dend_list = {"Completo": hc1, "Promedio": hc2}
pd.DataFrame({nombre: cophenet(arbol) for nombre, arbol in dend_list.items()}).corr()

#### Cuatro dendrogramas: completo, promedio, Ward y simple

In [ ]:
hc3 = linkage(d, method="ward")
hc4 = linkage(d, method="single")
dend_list.update({"Ward": hc3, "Simple": hc4})
cor_dend = pd.DataFrame({nombre: cophenet(arbol) for nombre, arbol in dend_list.items()}).corr()
cor_dend

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for i in range(len(cor_dend)):
    for j in range(i+1):
        valor = cor_dend.iloc[i, j]
        _ = ax.add_patch(Circle((j, i), 0.42, edgecolor="lightgrey", facecolor="white"))
        _ = ax.add_patch(Wedge((j, i), 0.42, 90, 90 + 360 * abs(valor),
                               facecolor="steelblue" if valor >= 0 else "tomato"))
        _ = ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=9)
_ = ax.set(xticks=range(4), yticks=range(4), xlim=(-0.6, 3.6), ylim=(3.6, -0.6),
           title="Correlaciones cofenéticas")
_ = ax.set_xticklabels(cor_dend.columns, rotation=45)
_ = ax.set_yticklabels(cor_dend.index)
_ = ax.set_aspect("equal")
fig.tight_layout()
plt.show()

### 9. Tendencia al agrupamiento: Hopkins

Usamos la convención en la que un valor próximo a **0** indica tendencia al agrupamiento.
Se contrastan distancias de observaciones reales y de puntos uniformes dentro del rango de los datos.

In [ ]:
def hopkins(datos, n=None, random_state=12345):
    a = np.asarray(datos, dtype=float)
    m = len(a) - 1 if n is None else int(n)
    if not 1 <= m < len(a):
        raise ValueError("El tamaño de muestra debe estar entre 1 y n-1.")
    rng = np.random.default_rng(random_state)
    indices = rng.choice(len(a), m, replace=False)
    reales = cdist(a[indices], a)
    reales[np.arange(m), indices] = np.inf
    w = reales.min(axis=1)
    uniformes = rng.uniform(a.min(axis=0), a.max(axis=0), size=(m, a.shape[1]))
    u = cdist(uniformes, a).min(axis=1)
    return w.sum() / (w.sum() + u.sum())

In [ ]:
hop = hopkins(iris2, n=len(iris2)-1, random_state=SEED)
hop

## Ejercicio 5-1 — Vinos

Aplica clustering jerárquico a `wine2.txt` para estudiar su tipología.

1. Lee los datos.
2. Haz una descriptiva sin escalar.
3. Estandariza los datos y dibuja un mapa de calor.
4. Examina las cuatro combinaciones de distancia **euclídea / Manhattan** y enlace **Ward / completo**.
   ¿Influye más la distancia o el método de agrupación?
5. Decide cuántos grupos sugiere cada árbol y realiza las cuatro particiones.
6. Escoge una partición y calcula su porcentaje de variabilidad explicada.
7. Describe las características relevantes de cada grupo. ¿En qué característica difieren menos?

**Archivos:** `wine2.txt` contiene las variables numéricas; la columna `Type` está en `wine.data.txt`.
Ambos están en `datos` y tienen el mismo orden de observaciones. Las etiquetas de tipo se reservan para interpretar los grupos.

**Ward:** la interpretación como incremento de varianza requiere distancia euclídea.
La actualización Ward sobre disimilitudes Manhattan puede explorarse usando una matriz condensada de distancias,
dejando explícita esa diferencia de interpretación.

Herramientas: `pd.read_csv`, `escalar`, `pdist`, `linkage`, `cut_tree` y `resumen_particion`.

In [ ]:
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos" / "wine2.txt").is_file():
        DATA_PATH = carpeta / "datos"
        break
else:
    raise FileNotFoundError("Abre el notebook dentro de la carpeta LaSalle_Estadistica.")

In [ ]:
# Ejercicio 5-1: tu solución

## K-means

### Objetos del ajuste

Agrupamos Iris en tres grupos. Exploramos las asignaciones, los centros y la descomposición de la inercia.

In [ ]:
km0 = ajustar_kmeans(iris2, k=3, n_init=1, random_state=SEED)
pd.Series(km0.labels_ + 1, index=iris2.index, name="Grupo")

In [ ]:
pd.DataFrame(km0.cluster_centers_, columns=iris2.columns, index=np.arange(1, 4))

In [ ]:
resumen_km0 = resumen_particion(iris2, km0.labels_)
resumen_km0.totss       # Inercia total
resumen_km0.withinss    # Inercia intra de cada grupo
resumen_km0.inertia_    # Inercia intra global
resumen_km0.betweenss   # Inercia entre grupos
resumen_km0.size        # Tamaños
km0.n_iter_            # Iteraciones

#### Variabilidad explicada

In [ ]:
resumen_km0.betweenss / resumen_km0.totss

### Número de grupos

#### Regla del codo: variabilidad explicada

In [ ]:
modelos_k = {k: ajustar_kmeans(iris2, k=k, n_init=10, random_state=SEED) for k in range(1, 11)}
VE = pd.Series({k: resumen_particion(iris2, modelos_k[k].labels_).betweenss / resumen_km0.totss
                for k in range(2, 11)}, name="Variabilidad explicada")
ax = VE.plot(marker="o")
_ = ax.set(xlabel="Número de grupos", ylabel="Variabilidad explicada")
plt.show()

#### Regla del codo: inercia dentro de los grupos

In [ ]:
inercia_k = pd.Series({k: modelo.inertia_ for k, modelo in modelos_k.items()})
ax = inercia_k.plot(marker="o")
_ = ax.set(xlabel="Número de grupos", ylabel="Inercia intra")
plt.show()

#### Número de grupos según los indicadores internos

In [ ]:
ncluster = evaluar_indices(iris2, metodo="kmeans", min_k=2, max_k=15, random_state=SEED)
ncluster.Best_n

In [ ]:
ax = ncluster.Votes.plot.bar(rot=0)
_ = ax.set(xlabel="Número de grupos", ylabel="Votos")
plt.show()

In [ ]:
indices_escalados = escalar(ncluster.All_index)
fig, ax = plt.subplots(figsize=(15, 6))
_ = sns.heatmap(indices_escalados, cmap="vlag", center=0, ax=ax)
_ = ax.set(title="Indicadores estandarizados", xlabel="Indicador", ylabel="Número de grupos")
fig.tight_layout()
plt.show()

### Representación: tres grupos

In [ ]:
km3 = ajustar_kmeans(iris2, k=3, n_init=10, random_state=SEED)
pares_coloreados(iris2, km3.labels_)

In [ ]:
pr_comp = PCA().fit(iris2)
coordenadas = pr_comp.transform(iris2)
x, y = coordenadas[:, 0], coordenadas[:, 1]
fig, ax = plt.subplots()
_ = ax.scatter(x, y, c=km3.labels_, cmap="tab10", vmin=0, vmax=9, s=24)
_ = ax.set(xlabel="CP1", ylabel="CP2", title="K-means: tres grupos")
plt.show()

In [ ]:
representar_clusters(iris2, km3.labels_, titulo="K-means: tres grupos y envolventes convexas")

#### Tres dimensiones: color por grupo y símbolo por especie. Cambia el ángulo de visión.

In [ ]:
def iris_3d(angulo):
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection="3d")
    for especie, marcador in enumerate(["o", "s", "^"]):
        mascara = dataset.target == especie
        ax.scatter(iris2.loc[mascara, "Sepal.Length"], iris2.loc[mascara, "Sepal.Width"],
                   iris2.loc[mascara, "Petal.Length"], c=km3.labels_[mascara], cmap="tab10",
                   vmin=0, vmax=9, marker=marcador, label=dataset.target_names[especie])
    for fila, grupo in zip(iris2.to_numpy(), km3.labels_):
        ax.plot([fila[0], fila[0]], [fila[1], fila[1]], [0, fila[2]],
                color=plt.get_cmap("tab10")(int(grupo)), alpha=0.15, linewidth=0.5)
    ax.set(xlabel="Sepal.Length", ylabel="Sepal.Width", zlabel="Petal.Length")
    ax.view_init(elev=20, azim=angulo)
    ax.legend()
    plt.show()

In [ ]:
control_3d = widgets.interactive(
    iris_3d,
    angulo=widgets.IntSlider(value=60, min=0, max=180, step=5, description="Ángulo", continuous_update=False),
)
display(control_3d)

### Representación: dos grupos

In [ ]:
km2 = ajustar_kmeans(iris2, k=2, n_init=10, random_state=SEED)
pares_coloreados(iris2, km2.labels_)

In [ ]:
fig, ax = plt.subplots()
_ = ax.scatter(x, y, c=km2.labels_, cmap="tab10", vmin=0, vmax=9, s=24)
_ = ax.set(xlabel="CP1", ylabel="CP2", title="K-means: dos grupos")
plt.show()

In [ ]:
representar_clusters(iris2, km2.labels_, titulo="K-means: dos grupos y envolventes convexas")

#### Explorar k, los inicios y el algoritmo

In [ ]:
def explorar_kmeans(k, inicios, algoritmo):
    modelo = ajustar_kmeans(iris2, k, n_init=inicios, algoritmo=algoritmo, random_state=SEED)
    representar_clusters(iris2, modelo.labels_, titulo=f"K-means: {k} grupos")
    resumen = resumen_particion(iris2, modelo.labels_)
    display(pd.Series({"Inercia intra": modelo.inertia_, "Variabilidad explicada": resumen.betweenss / resumen.totss,
                       "Silhouette": silhouette_score(iris2, modelo.labels_)}))

In [ ]:
control_kmeans = widgets.interactive(
    explorar_kmeans,
    k=widgets.IntSlider(value=3, min=2, max=8, description="Grupos", continuous_update=False),
    inicios=widgets.SelectionSlider(options=[1, 5, 10], value=10, description="Inicios", continuous_update=False),
    algoritmo=widgets.Dropdown(options=[("Hartigan–Wong", "hartigan-wong"), ("Lloyd", "lloyd")], description="Algoritmo"),
)
display(control_kmeans)

## K-medoides

PAM utiliza observaciones reales como centros. La fase BUILD escoge los medoides iniciales
y la fase SWAP busca intercambios que reduzcan la suma de distancias.

In [ ]:
def pam(datos, k, distancias=None, max_iter=100):
    a = np.asarray(datos, dtype=float)
    D = squareform(pdist(a)) if distancias is None else np.asarray(distancias)
    if not 1 <= k < len(a):
        raise ValueError("k debe ser menor que el número de observaciones.")
    # BUILD: añadir observaciones reales que reduzcan la distancia total.
    medoides = [int(D.sum(axis=0).argmin())]
    while len(medoides) < k:
        cercana = D[:, medoides].min(axis=1)
        costes = np.minimum(D, cercana[:, None]).sum(axis=0)
        costes[medoides] = np.inf
        medoides.append(int(costes.argmin()))
    # SWAP: buscar el mejor intercambio de un medoide por otra observación.
    historial = []
    for iteracion in range(max_iter):
        actuales = D[:, medoides]
        asignacion = actuales.argmin(axis=1)
        primera = actuales.min(axis=1)
        segunda = np.partition(actuales, 1, axis=1)[:, 1] if k > 1 else np.full(len(a), np.inf)
        coste_actual = primera.sum()
        historial.append(coste_actual)
        mejor = (coste_actual, None, None)
        for j in range(k):
            base = np.where(asignacion == j, segunda, primera)
            costes = np.minimum(D, base[:, None]).sum(axis=0)
            costes[medoides] = np.inf
            candidato = int(costes.argmin())
            if costes[candidato] < mejor[0] - 1e-9:
                mejor = (costes[candidato], j, candidato)
        if mejor[1] is None:
            break
        medoides[mejor[1]] = mejor[2]
    etiquetas = D[:, medoides].argmin(axis=1)
    return SimpleNamespace(labels_=etiquetas, medoid_indices_=np.array(medoides),
                           cluster_centers_=a[medoides], inertia_=D[:, medoides].min(axis=1).sum(),
                           history_=np.array(historial), n_iter_=iteracion + 1)

In [ ]:
kmedoids3 = pam(iris2, k=3)
representar_clusters(iris2, kmedoids3.labels_, titulo="PAM: tres grupos")

#### Similitud entre las particiones de k-means y PAM: índice de Rand ajustado

In [ ]:
adjusted_rand_score(km3.labels_, kmedoids3.labels_)

#### Similitud entre k-means y las especies originales

In [ ]:
adjusted_rand_score(km3.labels_, iris["Species"])

## Grupos no convexos

Generamos 500 observaciones en un disco de radio 1 y otras 500 en una corona de radios entre 2 y 3.
El radio se obtiene de una distribución uniforme, tal como plantea el ejemplo.

In [ ]:
n = 500
rng = np.random.default_rng(SEED)
r1 = rng.uniform(0, 1, n)
alfa1 = rng.uniform(0, 2 * np.pi, n)
x1, y1 = r1 * np.cos(alfa1), r1 * np.sin(alfa1)
r2 = rng.uniform(2, 3, n)
alfa2 = rng.uniform(0, 2 * np.pi, n)
x2, y2 = r2 * np.cos(alfa2), r2 * np.sin(alfa2)
df_anillos = pd.DataFrame({"x": np.r_[x1, x2], "y": np.r_[y1, y2]})
df_anillos_escalado = escalar(df_anillos)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
_ = ax.scatter(df_anillos["x"], df_anillos["y"], s=10)
_ = ax.set(xlabel="x", ylabel="y", title="Disco y corona circular")
_ = ax.set_aspect("equal")
plt.show()

In [ ]:
km_anillos = ajustar_kmeans(df_anillos_escalado, k=2, n_init=1, random_state=SEED)
fig, ax = plt.subplots(figsize=(6, 6))
_ = ax.scatter(df_anillos["x"], df_anillos["y"], c=km_anillos.labels_, cmap="tab10", vmin=0, vmax=9, s=10)
_ = ax.set(title="K-means en grupos no convexos", xlabel="x", ylabel="y")
_ = ax.set_aspect("equal")
plt.show()

### K-means con kernel

El kernel gaussiano define similitudes en otro espacio. K-means actualiza los grupos usando distancias
a sus centros en ese espacio, calculadas a partir de la matriz del kernel.
La inicialización utiliza una representación espectral del kernel.
El control gamma permite explorar cómo cambia la escala de semejanza.

In [ ]:
def kernel_kmeans(datos, k=2, gamma=2.0, random_state=12345, max_iter=100):
    a = np.asarray(datos, dtype=float)
    K = np.exp(-gamma * squareform(pdist(a, metric="sqeuclidean")))
    pesos = np.sqrt(K.sum(axis=1))
    normalizada = K / np.outer(pesos, pesos)
    with threadpool_limits(limits=1):
        _, vectores = eigh(normalizada, subset_by_index=[len(a) - k, len(a) - 1])
        vectores = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)
        etiquetas = KMeans(k, n_init=10, random_state=random_state).fit_predict(vectores)
    historial = []
    for iteracion in range(max_iter):
        distancias = np.empty((len(a), k))
        for grupo in range(k):
            miembros = np.flatnonzero(etiquetas == grupo)
            if not len(miembros):
                raise ValueError("El kernel ha producido un grupo vacío; cambia gamma.")
            distancias[:, grupo] = np.diag(K) - 2 * K[:, miembros].mean(axis=1) + K[np.ix_(miembros, miembros)].mean()
        nuevas = distancias.argmin(axis=1)
        historial.append(float(distancias[np.arange(len(a)), etiquetas].sum()))
        if np.array_equal(nuevas, etiquetas):
            break
        if len(np.unique(nuevas)) < k:
            break
        etiquetas = nuevas
    return SimpleNamespace(labels_=etiquetas, history_=np.array(historial), n_iter_=iteracion + 1)

In [ ]:
kk = kernel_kmeans(df_anillos_escalado, k=2, gamma=2.0, random_state=SEED)

def explorar_kernel(gamma):
    resultado = kernel_kmeans(df_anillos_escalado, k=2, gamma=gamma, random_state=SEED)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df_anillos["x"], df_anillos["y"], c=resultado.labels_, cmap="tab10", vmin=0, vmax=9, s=10)
    ax.set(xlabel="x", ylabel="y", title=f"K-means con kernel gaussiano: gamma = {gamma:.2f}")
    ax.set_aspect("equal")
    plt.show()

In [ ]:
control_kernel = widgets.interactive(
    explorar_kernel,
    gamma=widgets.FloatSlider(value=2.0, min=0.5, max=4, step=0.25, description="Gamma", continuous_update=False),
)
display(control_kernel)

## Valores atípicos

Generamos 1000 observaciones en dos grupos y tres observaciones alejadas, alrededor de 100000 en ambos ejes.
Representamos los datos con ejes logarítmicos y aplicamos k-means, PAM y CLARA.

CLARA ajusta PAM sobre muestras y evalúa sus medoides en el conjunto completo.

In [ ]:
def clara(datos, k, samples=5, sampsize=None, random_state=12345):
    a = np.asarray(datos, dtype=float)
    rng = np.random.default_rng(random_state)
    tamano = min(len(a), 40 + 2 * k) if sampsize is None else min(len(a), sampsize)
    if tamano <= k:
        raise ValueError("El tamaño de muestra debe ser mayor que k.")
    mejor = None
    for muestra in range(samples):
        conservar = np.array([], dtype=int) if mejor is None else mejor.medoid_indices_
        candidatos = np.setdiff1d(np.arange(len(a)), conservar)
        indices = np.r_[conservar, rng.choice(candidatos, tamano - len(conservar), replace=False)]
        local = pam(a[indices], k)
        medoides = indices[local.medoid_indices_]
        distancias = cdist(a, a[medoides])
        coste = distancias.min(axis=1).sum()
        if mejor is None or coste < mejor.inertia_:
            mejor = SimpleNamespace(labels_=distancias.argmin(axis=1), medoid_indices_=medoides,
                                     cluster_centers_=a[medoides], inertia_=coste)
    return mejor

In [ ]:
def generar_atipicos(nout=3):
    rng = np.random.default_rng(SEED)
    n = 1000
    x1 = np.r_[rng.normal(10, 1, n//2), rng.normal(100, 1, n//2), rng.normal(10**5, 1, nout)]
    x2 = np.r_[rng.normal(10, 1, n), rng.normal(10**5, 1, nout)]
    return pd.DataFrame({"x1": x1, "x2": x2})

df_atipicos = generar_atipicos(nout=3)
km_atipicos = ajustar_kmeans(df_atipicos, k=2, n_init=1, random_state=SEED)
pam_atipicos = pam(df_atipicos, k=2)
clara_atipicos = clara(df_atipicos, k=2, random_state=SEED)

In [ ]:
def dibujar_atipicos(tabla, modelos):
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, (nombre, grupos) in zip(axes.flat, modelos.items()):
        ax.scatter(tabla["x1"], tabla["x2"], c=grupos if grupos is not None else "steelblue",
                   cmap="tab10" if grupos is not None else None, s=14,
                   vmin=0 if grupos is not None else None, vmax=9 if grupos is not None else None)
        ax.set(xscale="log", yscale="log", xlabel="x1", ylabel="x2", title=nombre)
    fig.tight_layout()
    plt.show()

dibujar_atipicos(df_atipicos, {"Datos": None, "K-means": km_atipicos.labels_,
                              "PAM": pam_atipicos.labels_, "CLARA": clara_atipicos.labels_})

#### Explorar el número de observaciones atípicas

In [ ]:
def explorar_atipicos(numero):
    tabla = generar_atipicos(nout=numero)
    km = ajustar_kmeans(tabla, k=2, n_init=1, random_state=SEED)
    med = pam(tabla, k=2)
    muestra = clara(tabla, k=2, random_state=SEED)
    dibujar_atipicos(tabla, {"Datos": None, "K-means": km.labels_, "PAM": med.labels_, "CLARA": muestra.labels_})

In [ ]:
control_atipicos = widgets.interactive(
    explorar_atipicos,
    numero=widgets.IntSlider(value=3, min=0, max=8, description="Atípicos", continuous_update=False),
)
display(control_atipicos)

## Ejercicio 5-2 — Clientes de un mayorista

Las seis variables representan el gasto anual en:

| Variable del archivo | Producto |
| --- | --- |
| `Fresh` | Productos frescos |
| `Milk` | Leche y derivados |
| `Grocery` | Comestibles |
| `Frozen` | Congelados |
| `Detergents_Paper` | Detergentes y papel |
| `Delicassen` | Productos de delicatessen |

### 1. Lectura

Lee `Wholesale customers data.csv`, en `DATA_PATH`. Comprueba su separador: el archivo contiene tabuladores.

In [ ]:
# Apartado 1: tu solución

### 2. Descriptiva y estandarización

Describe los datos y estandarízalos. Valora si conviene transformar primero con `log(1 + x)`
para admitir valores cero y después estandarizar.

In [ ]:
# Apartado 2: tu solución

### 3. Número de grupos

Decide el número de grupos con la regla del codo o los indicadores de `evaluar_indices`.
¿Cuántos grupos sugieren los datos?

In [ ]:
# Apartado 3: tu solución

### 4. Cuatro ajustes de k-means

Estudia estos cuatro sistemas:

1. Lloyd con un inicio.
2. Hartigan–Wong con un inicio.
3. Hartigan–Wong con diez inicios.
4. K-means iniciado en los centros obtenidos de una partición jerárquica.

Para **5, 6, 7 y 8 grupos**, calcula el índice de Dunn y la silhouette media.

- Según cada índice, ¿cuántos grupos escogerías entre 5 y 8?
- ¿Depende la decisión del sistema empleado? ¿Depende del índice?

Herramientas: `ajustar_kmeans(..., algoritmo="lloyd" o "hartigan-wong", n_init=...)`,
`ajustar_kmeans(..., init=centros)`, `indice_dunn(D, grupos)` y
`silhouette_score(D, grupos, metric="precomputed")`.

In [ ]:
# Plantilla orientativa: completa los argumentos y los cálculos.
# DUN = pd.DataFrame(index=range(5, 9), columns=range(1, 5), dtype=float)
# SIL = pd.DataFrame(index=range(5, 9), columns=range(1, 5), dtype=float)
# D = squareform(pdist(datos2))
# for k in range(5, 9):
#     ...  # Obtener la jerarquía, cortarla y calcular los centros
#     ...  # Ajustar los cuatro sistemas
#     ...  # Calcular Dunn y silhouette para cada ajuste
# DUN.round(3)
# SIL.round(3)

### 5. Similitud de las particiones

Para **2** y **5 grupos**, calcula el índice de Rand ajustado entre Lloyd con un inicio
y Hartigan–Wong con diez inicios. Usa la semilla **1234**.

Herramienta: `adjusted_rand_score(grupos_a, grupos_b)`.

In [ ]:
# Apartado 5: tu solución

### 6. Describir los grupos

Describe los perfiles de los grupos de la partición escogida.
Puedes representar las variables mediante diagramas de caja separados por grupo.

In [ ]:
# Apartado 6: tu solución

### Documentación

- [Clustering jerárquico en SciPy](https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html).
- [K-means en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).
- [Silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).
- [Selección del número de grupos mediante indicadores internos](https://www.jstatsoft.org/article/view/v061i06).